# English → French NMT (LSTM encoder-decoder) — Colab runner
**Before you start:** `Runtime ▸ Change runtime type ▸ T4 GPU`.

Put these two files in a Drive folder called `NMT` (My Drive ▸ NMT):
1. `English_French_NMT_Project.zip` (this project)
2. `en-fr.zip` (the IWSLT 2017 dataset you were given)

Checkpoints are saved to Drive, so you can resume if Colab disconnects (`--resume`).

In [ ]:
!nvidia-smi -L

In [ ]:
import os, zipfile, shutil
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/NMT'
CKPT = f'{BASE}/checkpoints'          # persists on Drive
RES  = f'{BASE}/results'
os.makedirs(CKPT, exist_ok=True); os.makedirs(RES, exist_ok=True)

# unzip the project into /content and put the dataset where the code expects it
zipfile.ZipFile(f'{BASE}/English_French_NMT_Project.zip').extractall('/content')
%cd /content/English_French_NMT_Project
os.makedirs('data', exist_ok=True)
shutil.copy(f'{BASE}/en-fr.zip', 'data/en-fr.zip')

In [ ]:
!pip install -q datasets tokenizers sacrebleu gradio matplotlib   # TensorFlow is already installed on Colab

## 1) Train
Defaults: batch 32, embedding 256, 512 LSTM units, 1 layer, lr 1e-3, 15 epochs, BPE vocab 10k.
Quick smoke test first (≈ 1 minute): `--epochs 1 --max_train_samples 5000`.
If the session dies, run this cell again with `--resume` added.

In [ ]:
!python train.py --ckpt_dir {CKPT} --results_dir {RES}
# !python train.py --ckpt_dir {CKPT} --results_dir {RES} --resume      # continue after a disconnect

## 2) Evaluate (validation loss + BLEU)

In [ ]:
!python evaluate.py --ckpt_dir {CKPT} --max_sents 1000 --beam 5

## 3) Translate

In [ ]:
!python translate.py --ckpt_dir {CKPT} --beam 5 --text "Hello, how are you?" "I would like a cup of coffee, please." "We are going to talk about climate change."

In [ ]:
from IPython.display import Image, display
display(Image(f'{RES}/loss_curve.png'))

## 4) Optional: web demo

In [ ]:
!python app.py --ckpt_dir {CKPT} --share